## Offline Evaluation

in this notebook, I aim to work on following ideas:

- [x] run experiments with different parameters
- [ ] list experiment artifacts, metrics and results in general
- [x] run A/B testing pipeline in the notebook by passing two different models loaded from the model registry, and splitted test data
- [x] run all models in an experiement against a part of data (here one of A_B_1 or A_B_2)  

In [51]:
dataset = "data/data_train_features_need_preprocessing_salary_less_than_500k_and_above_1k.parquet"
dataset_name = "linkedin_joblist_salary_less_than_500k_and_above_1k"

flow = "pipelines/experiment_1_decision_tree_fault_tolerance.py"

### Split dataset

In [52]:
import pandas as pd
from sklearn.model_selection import train_test_split

seed = 42

df = pd.read_parquet(dataset)

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=seed)
A_B_1, A_B_2 = train_test_split(temp_df, test_size=0.5, random_state=seed)

train_df.to_parquet('data/train.parquet', index=False)
A_B_1.to_parquet('data/A_B_1.parquet', index=False)
A_B_2.to_parquet('data/A_B_2.parquet', index=False)

In [48]:
dataset_train = "data/train.parquet"
dataset_A_B_1 = "data/A_B_1.parquet"
dataset_A_B_2 = "data/A_B_2.parquet"

### Run experiments with different parameters

In [45]:
from metaflow import Runner

In [46]:
# it is set by default
# max_depth = 30
run_id="A_B_test_control"
model_name="A_B_control_model"
with Runner(flow).run(dataset='data/train.parquet', dataset_name=dataset_name, model_name=model_name) as running:
    print(f'{running.run} finished')

Metaflow 2.15.9 executing DTRFlow for user:mahdikhashan
Validating your flow...
    The graph looks good!
Running pylint...
    Pylint not found, so extra checks are disabled.
Including file data/train.parquet of size 50MB 
Attempting to use MLflow Tracking URI: None
2025-06-18 18:10:08.336 Workflow starting (run-id 1750263006872182):
2025-06-18 18:10:08.350 [1750263006872182/start/1 (pid 25606)] Task is starting.
2025-06-18 18:10:08.760 [1750263006872182/start/1 (pid 25606)] Attempting to use MLflow Tracking URI: None
2025-06-18 18:10:08.829 [1750263006872182/start/1 (pid 25606)] Starting flow 'DTRFlow' with run ID '1750263006872182'
2025-06-18 18:10:08.893 [1750263006872182/start/1 (pid 25606)] Task finished successfully.
2025-06-18 18:10:08.899 [1750263006872182/load_dataset/2 (pid 25613)] Task is starting.
2025-06-18 18:10:09.277 [1750263006872182/load_dataset/2 (pid 25613)] Attempting to use MLflow Tracking URI: None
2025-06-18 18:10:09.531 [1750263006872182/load_dataset/2 (pid 25

In [47]:
# second model training with a different parameter
max_depth=5
model_name="A_B_second_model"
with Runner(flow).run(dataset='data/train.parquet', dataset_name=dataset_name, max_depth=max_depth, model_name=model_name) as running:
    print(f'{running.run} finished')

Metaflow 2.15.9 executing DTRFlow for user:mahdikhashan
Validating your flow...
    The graph looks good!
Running pylint...
    Pylint not found, so extra checks are disabled.
Including file data/train.parquet of size 50MB 
Attempting to use MLflow Tracking URI: None
2025-06-18 18:13:20.178 Workflow starting (run-id 1750263198685669):
2025-06-18 18:13:20.190 [1750263198685669/start/1 (pid 25799)] Task is starting.
2025-06-18 18:13:20.581 [1750263198685669/start/1 (pid 25799)] Attempting to use MLflow Tracking URI: None
2025-06-18 18:13:20.648 [1750263198685669/start/1 (pid 25799)] Starting flow 'DTRFlow' with run ID '1750263198685669'
2025-06-18 18:13:20.715 [1750263198685669/start/1 (pid 25799)] Task finished successfully.
2025-06-18 18:13:20.720 [1750263198685669/load_dataset/2 (pid 25806)] Task is starting.
2025-06-18 18:13:21.193 [1750263198685669/load_dataset/2 (pid 25806)] Attempting to use MLflow Tracking URI: None
2025-06-18 18:13:21.468 [1750263198685669/load_dataset/2 (pid 25

### List artifacts of experiments

In [ ]:
# TODO(mahdi): ...

### run A/B test

In [55]:
a_b_test_flow = "pipelines/evaluate_A_B.py"
model_a_uri="models:/A_B_second_model/2"
model_b_uri="models:/A_B_control_model/1"
dataset_a_b = "data/A_B_1.parquet"
with Runner(a_b_test_flow).run(dataset=dataset_a_b, model_a_uri=model_a_uri, model_b_uri=model_b_uri) as running:
    print(f'{running.run} finished')

Metaflow 2.15.9 executing Evaluate_A_B_Flow for user:mahdikhashan
Validating your flow...
    The graph looks good!
Running pylint...
    Pylint not found, so extra checks are disabled.
Including file data/A_B_1.parquet of size 6MB 
2025-06-18 20:42:43.173 Workflow starting (run-id 1750272162974094):
2025-06-18 20:42:43.182 [1750272162974094/start/1 (pid 48502)] Task is starting.
2025-06-18 20:42:43.377 [1750272162974094/start/1 (pid 48502)] Task finished successfully.
2025-06-18 20:42:43.382 [1750272162974094/load_dataset/2 (pid 48509)] Task is starting.
2025-06-18 20:42:44.284 [1750272162974094/load_dataset/2 (pid 48509)] Task finished successfully.
2025-06-18 20:42:44.290 [1750272162974094/validate_dataset/3 (pid 48516)] Task is starting.
2025-06-18 20:42:44.822 [1750272162974094/validate_dataset/3 (pid 48516)] ============================= test session starts ==============================
2025-06-18 20:42:44.823 [1750272162974094/validate_dataset/3 (pid 48516)] platform darwin -- 

### run all models in an experiement (based on mlflow)

In [56]:
experiment_test_flow = "pipelines/evaluate_experiment.py"
experiment_name = "my-experiment-fault-tolerance"
dataset_a_b = "data/A_B_1.parquet"
with Runner(experiment_test_flow).run(dataset=dataset_a_b, experiment_name=experiment_name) as running:
    print(f'{running.run} finished')

Metaflow 2.15.9 executing EvaluateExperimentFlow for user:mahdikhashan
Validating your flow...
    The graph looks good!
Running pylint...
    Pylint not found, so extra checks are disabled.
Including file data/A_B_1.parquet of size 6MB 
2025-06-18 20:43:36.917 Workflow starting (run-id 1750272216734687):
2025-06-18 20:43:36.925 [1750272216734687/start/1 (pid 48612)] Task is starting.
2025-06-18 20:43:37.905 [1750272216734687/start/1 (pid 48612)] Task finished successfully.
2025-06-18 20:43:37.911 [1750272216734687/load_dataset/2 (pid 48619)] Task is starting.
2025-06-18 20:43:38.449 [1750272216734687/load_dataset/2 (pid 48619)] Task finished successfully.
2025-06-18 20:43:38.453 [1750272216734687/validate_dataset/3 (pid 48626)] Task is starting.
2025-06-18 20:43:38.962 [1750272216734687/validate_dataset/3 (pid 48626)] ============================= test session starts ==============================
2025-06-18 20:43:38.962 [1750272216734687/validate_dataset/3 (pid 48626)] platform darwi